In [ ]:
# Libs

from prophet import Prophet
from sqlalchemy import create_engine
import pandas as pd

# Database connection

engine = create_engine(
    "postgresql+psycopg2://postgres:@localhost:5432/Online Retail II UCI"
)

In [ ]:
# Total business revenue per day (all valid stock code types)

df_business = pd.read_sql("""
    SELECT
        invoice_date::date AS ds,
        SUM(revenue) AS y
    FROM fact_transactions
    WHERE is_return = false
    GROUP BY invoice_date::date
    ORDER BY invoice_date::date ASC
""", engine)

# Product revenue per day (valid_product only)

df_products = pd.read_sql("""
    SELECT 
        invoice_date::date AS ds,
        SUM(revenue) AS y	
    FROM fact_transactions ft
    INNER JOIN dim_products dp
    ON ft.stock_code = dp.stock_code
    WHERE most_freq_sctype = 'valid_product'
    GROUP BY invoice_date::date
    ORDER BY invoice_date::date ASC
""", engine)


Prophet is a library created by Meta that predicts future values from a time series. What makes it powerful is that it automatically detects:
- General trend → revenue rises or falls over the long term
- Seasonality → recurring patterns (weekly, monthly, yearly)
- Special days → Black Friday, Christmas...

Prophet is very strict about column names; it only accepts:
- ds → the date column (ds = datestamp)
- y → the column to be predicted (y = your value)

Prophet works in three very simple steps:
1. Créer le modèle → m = Prophet()
2. Entraîner → m.fit(df)
3. Prédire → m.predict(future)

In [ ]:
# Reusable Prophet pipeline
# periods : number of future periods to forecast (days or weeks depending on input granularity)
# Returns forecast columns only : ds, yhat, yhat_lower, yhat_upper

def run_prophet(df, days):
    m = Prophet()
    m.fit(df)
    future = m.make_future_dataframe(periods=days)
    df = m.predict(future)
    return df[["ds", "yhat", "yhat_lower", "yhat_upper"]], m 

In [ ]:
# Train Prophet on both datasets — 365 days horizon

df_business_fc, m_business = run_prophet(df_business, 365)
df_products_fc, m_products = run_prophet(df_products, 365)

You don’t need to know them all! The four important ones are:
- ds → the date
- yhat → the prediction (y hat = estimated value)
- yhat_lower → the lower limit of the confidence interval
- yhat_upper → the upper limit of the confidence interval

A confidence interval is Prophet telling you, “I’m X% certain that the true value will lie between yhat_lower and yhat_upper”.
By default, Prophet uses 80% this is its default value, which you can change when creating the model:

-- m = Prophet(interval_width=0.95)  # 95% de confiance --

In [ ]:
# Convert ds to datetime before merging
# PostgreSQL returns ds as object — Prophet requires datetime64

df_business['ds'] = pd.to_datetime(df_business['ds'])
df_products['ds'] = pd.to_datetime(df_products['ds'])

In [ ]:
# Merge actual values (y) into forecast DataFrames for MAE evaluation

df_business_fc = df_business_fc.merge(df_business, how='left', left_on='ds', right_on='ds')
df_products_fc = df_products_fc.merge(df_products, how='left', left_on='ds', right_on='ds')

In [ ]:
# Visual check — does the forecast look consistent with historical data ?

m_business.plot(df_business_fc)

In [ ]:
m_products.plot(df_products_fc)

In [ ]:
# MAE evaluation — computed on historical data only (non-null y)
# Returns mae (mean absolute error) and mean (average actual revenue)

def calculate_mae(df):
    df_past = df[df['y'].notnull()]
    mae = abs(df_past['y'] - df_past['yhat']).mean()
    mean = df_past['y'].mean()
    return mae, mean

In [ ]:
mae_business, mean_business = calculate_mae(df_business_fc)
print(f'Business MAE is {round(mae_business,2)} & MEAN is {round(mean_business,2)}')

In [ ]:
mae_products, mean_products = calculate_mae(df_products_fc)
print(f'Products MAE is {round(mae_products,2)} & MEAN is {round(mean_products,2)}')

Daily e-commerce data is, by its very nature, highly volatile. Let's try to compare on a weekly basis:

In [ ]:
# Product revenue aggregated by week

df_products_weekly = pd.read_sql("""
    SELECT 
        DATE_TRUNC('week', invoice_date)::date AS ds,
        SUM(revenue) AS y	
    FROM fact_transactions ft
    INNER JOIN dim_products dp ON ft.stock_code = dp.stock_code
    WHERE most_freq_sctype = 'valid_product'
    GROUP BY DATE_TRUNC('week', invoice_date)::date
    ORDER BY DATE_TRUNC('week', invoice_date)::date ASC
""", engine)

In [ ]:
# 52 periods = 1 year horizon at weekly granularity

df_products_weekly_fc, m_products_weekly = run_prophet(df_products_weekly, 52)

In [ ]:
df_products_weekly['ds'] = pd.to_datetime(df_products_weekly['ds'])

In [ ]:
df_products_weekly_fc = df_products_weekly_fc.merge(df_products_weekly, how='left', left_on='ds', right_on='ds')

In [ ]:
m_products_weekly.plot(df_products_weekly_fc)

In [ ]:
mae_products_weekly, mean_products_weekly = calculate_mae(df_products_weekly_fc)
print(f'Products (weekly) MAE is {round(mae_products_weekly,2)} & MEAN is {round(mean_products_weekly,2)}')

In [ ]:
df_products_weekly_fc['model'] = 'Prophet'
df_products_weekly_fc['granularity'] = 'weekly'

In [ ]:
# Append to forecast_results — if_exists='append' preserves existing rows
# index=False prevents pandas from writing the DataFrame index as a column

df_products_weekly_fc.to_sql(
    name='forecast_results_products_prophet',
    con=engine,
    if_exists='append',
    index=False
)

In [ ]:
# Business revenue aggregated by week

df_business_weekly = pd.read_sql("""
SELECT 
    DATE_TRUNC('week', invoice_date)::date AS ds,
    SUM(revenue) AS y	
FROM fact_transactions ft
INNER JOIN dim_products dp ON ft.stock_code = dp.stock_code
WHERE most_freq_sctype LIKE 'valid%%'
GROUP BY DATE_TRUNC('week', invoice_date)::date
ORDER BY DATE_TRUNC('week', invoice_date)::date ASC
""", engine)

In [ ]:
# 52 periods = 1 year horizon at weekly granularity

df_business_weekly_fc, m_business_weekly = run_prophet(df_products_weekly, 52)

In [ ]:
df_business_weekly['ds'] = pd.to_datetime(df_business_weekly['ds'])

In [ ]:
df_business_weekly_fc = df_business_weekly_fc.merge(df_business_weekly, how='left', left_on='ds', right_on='ds')

In [ ]:
m_business_weekly.plot(df_products_weekly_fc)

In [ ]:
mae_business_weekly, mean_business_weekly = calculate_mae(df_products_weekly_fc)
print(f'Business (weekly) MAE is {round(mae_business_weekly,2)} & MEAN is {round(mean_business_weekly,2)}')

In [51]:
df_business_weekly_fc['model'] = 'Prophet'
df_business_weekly_fc['granularity'] = 'weekly'

In [52]:
df_business_weekly_fc.to_sql(
    name='forecast_results_business_prophet',
    con=engine,
    if_exists='append',
    index=False
)

156